# FIT5196 Assessment 1 - Solution Notebook — Task 1 Version

**Group:** Group050


**Members:** <br>
Yu Wang ID:  ; <br>
Xingao Zhan ID: 36354775 <br>
Keshu Zhang ID:   ; <br>
Qingzhuo Zhao ID: 26662841

**Stage label:** Task 1 — structured parsing, source profiling and source-to-target mapping  
**Version date:** 2026-08-22  
**Status:** Task 1 complete — source-profile checks PASS; mapping audit 12/12 PASS  
**Scope:** This stage version completes Task 1 only. Tasks 2–6 remain outside this checkpoint.


## 0. Configuration and reproducibility

Keep all configurable paths in this section. The final notebook must run with
**Restart and Run All** without manual file edits or network access.


In [1]:
from pathlib import Path
import sys

GROUP_ID = "Group050"
PROJECT_DIR = Path.cwd()

# Official default: support files and raw_input are beside the notebook.
# Development fallback: keep the supplied package read-only in the sibling
# Group050_A1 folder while all generated and edited files stay here.
PACKAGE_DIR = PROJECT_DIR
LOCAL_PACKAGE_DIR = PROJECT_DIR.parent / "Group050_A1"
if not (PACKAGE_DIR / "raw_input").is_dir() and (LOCAL_PACKAGE_DIR / "raw_input").is_dir():
    PACKAGE_DIR = LOCAL_PACKAGE_DIR

INPUT_DIR = PACKAGE_DIR / "raw_input"
OUTPUT_DIR = PROJECT_DIR / "outputs"
TEMPLATE_DIR = PACKAGE_DIR / "templates"

JSON_PATH = INPUT_DIR / f"{GROUP_ID}_commerce.json"
XML_PATH = INPUT_DIR / f"{GROUP_ID}_operations.xml"
DATA_DICTIONARY_PATH = PACKAGE_DIR / "public_data_dictionary.csv"
MAPPING_TEMPLATE_PATH = TEMPLATE_DIR / "A1_source_to_target_mapping_template.csv"
MAPPING_OUTPUT_PATH = PROJECT_DIR / "Task 01 Data" / f"{GROUP_ID}_source_to_target_mapping.csv"
PUBLIC_TEXT_TESTS_PATH = TEMPLATE_DIR / "A1_public_text_test_cases.csv"

required_paths = {
    "JSON source": JSON_PATH,
    "XML source": XML_PATH,
    "public data dictionary": DATA_DICTIONARY_PATH,
    "mapping template": MAPPING_TEMPLATE_PATH,
    "public text tests": PUBLIC_TEXT_TESTS_PATH,
}
missing_paths = {
    label: path for label, path in required_paths.items() if not path.is_file()
}
if missing_paths:
    missing_text = "\n".join(
        f"- {label}: {path}" for label, path in missing_paths.items()
    )
    raise FileNotFoundError(f"Required assignment files are missing:\n{missing_text}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

package_display = "." if PACKAGE_DIR == PROJECT_DIR else "../Group050_A1"
print("Working directory: .")
print(f"Read-only package directory: {package_display}")
print(f"JSON source: {JSON_PATH.name}")
print(f"XML source: {XML_PATH.name}")
print("Output directory: outputs")

Working directory: .
Read-only package directory: .
JSON source: Group050_commerce.json
XML source: Group050_operations.xml
Output directory: outputs


### 0.1 Environment and dependencies

Import the libraries used by your submitted workflow. Record non-standard
dependencies in `requirements.txt`.


In [2]:
# EVIDENCE: SEC-0.1-ENVIRONMENT
import json
import platform
import sys
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd

def show(frame, title=None):
    if title:
        print(f"\n{title}")
    print(frame.to_string(index=False))

environment = pd.DataFrame(
    [
        {"component": "Python", "version": platform.python_version()},
        {"component": "pandas", "version": pd.__version__},
        {"component": "NumPy", "version": np.__version__},
        {"component": "XML parser", "version": "xml.etree.ElementTree (stdlib)"},
        {"component": "JSON parser", "version": "json (stdlib)"},
    ]
)
show(environment, "Environment and dependencies")



Environment and dependencies
  component                        version
     Python                        3.12.13
     pandas                          2.2.3
      NumPy                          2.1.3
 XML parser xml.etree.ElementTree (stdlib)
JSON parser                  json (stdlib)


## 1. Parse and profile the two sources

Use structured JSON and XML parsers. Record source grains, nested/repeated
structures, candidate keys, formats, missing-value conventions and evidence of
within-source or cross-source overlap.


### 1.1 JSON structure and profile


In [3]:
# EVIDENCE: SEC-1.1-JSON-PROFILE
with JSON_PATH.open(encoding="utf-8") as handle:
    json_source = json.load(handle)

json_customers = json_source["customerProfiles"]
json_orders = json_source["orders"]
json_items = [item for order in json_orders for item in order["shoppingCart"]]
json_deliveries = [order["delivery"] for order in json_orders]
json_reviews = json_source["productReviews"]

def profile_records(source_collection, grain, records, key_field):
    keys = [record.get(key_field) for record in records]
    non_missing = [key for key in keys if key not in (None, "")]
    frequencies = Counter(non_missing)
    fingerprints = Counter(
        json.dumps(record, sort_keys=True, ensure_ascii=False) for record in records
    )
    return {
        "source_collection": source_collection,
        "grain": grain,
        "candidate_key": key_field,
        "rows": len(records),
        "missing_key": len(keys) - len(non_missing),
        "unique_key": len(frequencies),
        "duplicate_key_groups": sum(count > 1 for count in frequencies.values()),
        "duplicate_extra_rows": len(non_missing) - len(frequencies),
        "exact_duplicate_groups": sum(count > 1 for count in fingerprints.values()),
        "exact_duplicate_extra_rows": len(records) - len(fingerprints),
    }

json_profile = pd.DataFrame(
    [
        profile_records("customerProfiles[]", "one customer profile", json_customers, "customerID"),
        profile_records("orders[]", "one source order", [order["header"] for order in json_orders], "orderID"),
        profile_records("orders[].shoppingCart[]", "one source order item", json_items, "orderItemID"),
        profile_records("orders[].delivery", "one source delivery", json_deliveries, "deliveryID"),
        profile_records("productReviews[]", "one source review", json_reviews, "reviewID"),
    ]
)
show(json_profile, "JSON collection profile")

json_structure = pd.DataFrame(
    [
        {"path": "customerProfiles[]", "representation": "repeated root array", "fields": ", ".join(json_customers[0].keys())},
        {"path": "orders[]", "representation": "repeated root array with header, shoppingCart[] and delivery", "fields": ", ".join(json_orders[0].keys())},
        {"path": "orders[].header", "representation": "nested object", "fields": ", ".join(json_orders[0]["header"].keys())},
        {"path": "orders[].shoppingCart[]", "representation": "nested repeated array", "fields": ", ".join(json_items[0].keys())},
        {"path": "orders[].delivery", "representation": "nested object", "fields": ", ".join(json_deliveries[0].keys())},
        {"path": "productReviews[]", "representation": "repeated root array", "fields": ", ".join(json_reviews[0].keys())},
    ]
)
show(json_structure, "JSON nesting and fields")

first_header = json_orders[0]["header"]
first_delivery = json_orders[0]["delivery"]
json_formats = pd.DataFrame(
    [
        {"concept": "timestamp", "path": "orders[].header.orderTimestamp", "example": repr(first_header["orderTimestamp"]), "Python type": type(first_header["orderTimestamp"]).__name__},
        {"concept": "date", "path": "orders[].delivery.dispatchDate", "example": repr(first_delivery["dispatchDate"]), "Python type": type(first_delivery["dispatchDate"]).__name__},
        {"concept": "boolean", "path": "orders[].header.expeditedDelivery", "example": repr(first_header["expeditedDelivery"]), "Python type": type(first_header["expeditedDelivery"]).__name__},
        {"concept": "currency value", "path": "orders[].header.orderPrice", "example": repr(first_header["orderPrice"]), "Python type": type(first_header["orderPrice"]).__name__},
        {"concept": "percentage points", "path": "orders[].header.couponDiscount", "example": repr(first_header["couponDiscount"]), "Python type": type(first_header["couponDiscount"]).__name__},
        {"concept": "missing optional string", "path": "orders[].header.couponCode", "example": "empty string count=" + str(sum(order["header"].get("couponCode") == "" for order in json_orders)), "Python type": "str"},
    ]
)
show(json_formats, "JSON source-format evidence")

json_collections = {
    "customerProfiles[]": json_customers,
    "orders[].header": [order["header"] for order in json_orders],
    "orders[].shoppingCart[]": json_items,
    "orders[].delivery": json_deliveries,
    "productReviews[]": json_reviews,
}
json_missing_rows = []
for source_collection, records in json_collections.items():
    fields = sorted({field for record in records for field in record})
    for field in fields:
        absent_count = sum(field not in record for record in records)
        null_count = sum(record.get(field) is None for record in records if field in record)
        empty_string_count = sum(record.get(field) == "" for record in records if field in record)
        if absent_count or null_count or empty_string_count:
            json_missing_rows.append(
                {
                    "source_collection": source_collection,
                    "field": field,
                    "absent_key": absent_count,
                    "JSON null": null_count,
                    "empty_string": empty_string_count,
                }
            )
json_missing_profile = pd.DataFrame(json_missing_rows)
show(json_missing_profile, "JSON missing-value conventions (non-zero counts only)")



JSON collection profile
      source_collection                 grain candidate_key  rows  missing_key  unique_key  duplicate_key_groups  duplicate_extra_rows  exact_duplicate_groups  exact_duplicate_extra_rows
     customerProfiles[]  one customer profile    customerID   500            0         500                     0                     0                       0                           0
               orders[]      one source order       orderID  2818            0        2750                    68                    68                      68                          68
orders[].shoppingCart[] one source order item   orderItemID  8884            0        8666                   218                   218                     218                         218
      orders[].delivery   one source delivery    deliveryID  2818            0        2750                    68                    68                      68                          68
       productReviews[]     one source r

### 1.2 XML structure and profile


In [4]:
# EVIDENCE: SEC-1.2-XML-PROFILE
xml_tree = ET.parse(XML_PATH)
xml_root = xml_tree.getroot()

xml_orders = xml_root.findall("./Orders/Order")
xml_headers = [order.find("Header") for order in xml_orders]
xml_items = xml_root.findall("./Orders/Order/Shopping_Cart/Item")
xml_deliveries = xml_root.findall("./Orders/Order/Delivery")
xml_products = xml_root.findall("./ProductCatalogue/Product")
xml_reviews = xml_root.findall("./ProductReviews/Review")
xml_warehouses = list(xml_root.find("WarehouseDirectory"))

def element_records(elements):
    return [{child.tag: child.text or "" for child in element} for element in elements]

xml_header_records = element_records(xml_headers)
xml_item_records = element_records(xml_items)
xml_delivery_records = element_records(xml_deliveries)
xml_product_records = element_records(xml_products)
xml_review_records = element_records(xml_reviews)
xml_warehouse_records = element_records(xml_warehouses)

xml_profile = pd.DataFrame(
    [
        profile_records("/OperationsExport/Orders/Order/Header", "one source order", xml_header_records, "Order_ID"),
        profile_records("/OperationsExport/Orders/Order/Shopping_Cart/Item", "one source order item", xml_item_records, "Order_Item_ID"),
        profile_records("/OperationsExport/Orders/Order/Delivery", "one source delivery", xml_delivery_records, "Delivery_ID"),
        profile_records("/OperationsExport/ProductCatalogue/Product", "one product", xml_product_records, "Product_ID"),
        profile_records("/OperationsExport/ProductReviews/Review", "one source review", xml_review_records, "Review_ID"),
        profile_records("/OperationsExport/WarehouseDirectory/*", "one warehouse reference", xml_warehouse_records, "Name"),
    ]
)
show(xml_profile, "XML collection profile")

xml_structure = pd.DataFrame(
    [
        {"path": "/OperationsExport", "representation": "root element", "child elements": ", ".join(child.tag for child in xml_root)},
        {"path": "/OperationsExport/Orders/Order", "representation": "repeated element with Header, Shopping_Cart and Delivery", "child elements": ", ".join(child.tag for child in xml_orders[0])},
        {"path": "/OperationsExport/Orders/Order/Shopping_Cart/Item", "representation": "nested repeated element", "child elements": ", ".join(xml_item_records[0].keys())},
        {"path": "/OperationsExport/ProductCatalogue/Product", "representation": "repeated element", "child elements": ", ".join(xml_product_records[0].keys())},
        {"path": "/OperationsExport/ProductReviews/Review", "representation": "repeated element", "child elements": ", ".join(xml_review_records[0].keys())},
        {"path": "/OperationsExport/WarehouseDirectory/*", "representation": "source-specific reference collection", "child elements": ", ".join(xml_warehouse_records[0].keys())},
    ]
)
show(xml_structure, "XML nesting and fields")

first_xml_header = xml_header_records[0]
first_xml_delivery = xml_delivery_records[0]
xml_formats = pd.DataFrame(
    [
        {"concept": "timestamp", "path": ".../Header/Order_Timestamp", "example": repr(first_xml_header["Order_Timestamp"]), "XML representation": "text"},
        {"concept": "date", "path": ".../Delivery/Dispatch_Date", "example": repr(first_xml_delivery["Dispatch_Date"]), "XML representation": "text"},
        {"concept": "boolean", "path": ".../Header/Expedited_Delivery", "example": repr(first_xml_header["Expedited_Delivery"]), "XML representation": "Y/N text"},
        {"concept": "currency value", "path": ".../Header/Order_Price", "example": repr(first_xml_header["Order_Price"]), "XML representation": "currency-labelled text"},
        {"concept": "percentage", "path": ".../Header/Coupon_Discount", "example": repr(first_xml_header["Coupon_Discount"]), "XML representation": "percent-labelled text"},
        {"concept": "missing optional string", "path": ".../Header/Coupon_Code", "example": "empty element count=" + str(sum(record["Coupon_Code"] == "" for record in xml_header_records)), "XML representation": "empty element"},
    ]
)
show(xml_formats, "XML source-format evidence")

xml_collections = {
    "/OperationsExport/Orders/Order/Header": xml_header_records,
    "/OperationsExport/Orders/Order/Shopping_Cart/Item": xml_item_records,
    "/OperationsExport/Orders/Order/Delivery": xml_delivery_records,
    "/OperationsExport/ProductCatalogue/Product": xml_product_records,
    "/OperationsExport/ProductReviews/Review": xml_review_records,
}
xml_missing_rows = []
for source_collection, records in xml_collections.items():
    fields = sorted({field for record in records for field in record})
    for field in fields:
        absent_element_count = sum(field not in record for record in records)
        empty_element_count = sum(record.get(field) == "" for record in records)
        if absent_element_count or empty_element_count:
            xml_missing_rows.append(
                {
                    "source_collection": source_collection,
                    "field": field,
                    "absent_element": absent_element_count,
                    "empty_element": empty_element_count,
                }
            )
xml_missing_profile = pd.DataFrame(xml_missing_rows)
show(xml_missing_profile, "XML missing-value conventions (non-zero counts only)")



XML collection profile
                                source_collection                   grain candidate_key  rows  missing_key  unique_key  duplicate_key_groups  duplicate_extra_rows  exact_duplicate_groups  exact_duplicate_extra_rows
            /OperationsExport/Orders/Order/Header        one source order      Order_ID  2818            0        2750                    68                    68                      68                          68
/OperationsExport/Orders/Order/Shopping_Cart/Item   one source order item Order_Item_ID  8833            0        8622                   211                   211                     211                         211
          /OperationsExport/Orders/Order/Delivery     one source delivery   Delivery_ID  2818            0        2750                    68                    68                      68                          68
       /OperationsExport/ProductCatalogue/Product             one product    Product_ID  1000            0        10

### 1.3 Source comparison and assumptions


In [5]:
# EVIDENCE: SEC-1.3-SOURCE-COMPARISON
json_key_sets = {
    "orders": {record["orderID"] for record in [order["header"] for order in json_orders]},
    "order_items": {record["orderItemID"] for record in json_items},
    "deliveries": {record["deliveryID"] for record in json_deliveries},
    "product_reviews": {record["reviewID"] for record in json_reviews},
}
xml_key_sets = {
    "orders": {record["Order_ID"] for record in xml_header_records},
    "order_items": {record["Order_Item_ID"] for record in xml_item_records},
    "deliveries": {record["Delivery_ID"] for record in xml_delivery_records},
    "product_reviews": {record["Review_ID"] for record in xml_review_records},
}

overlap_rows = []
for entity in json_key_sets:
    json_keys = json_key_sets[entity]
    xml_keys = xml_key_sets[entity]
    overlap = json_keys & xml_keys
    overlap_rows.append(
        {
            "entity": entity,
            "JSON unique keys": len(json_keys),
            "XML unique keys": len(xml_keys),
            "cross-source overlap": len(overlap),
            "JSON only": len(json_keys - xml_keys),
            "XML only": len(xml_keys - json_keys),
            "union before field reconciliation": len(json_keys | xml_keys),
        }
    )
overlap_profile = pd.DataFrame(overlap_rows)
show(overlap_profile, "Cross-source key overlap")

combined_repeat_profile = pd.concat(
    [
        json_profile.assign(source="JSON"),
        xml_profile.assign(source="XML"),
    ],
    ignore_index=True,
)[
    ["source", "source_collection", "grain", "candidate_key", "rows", "missing_key", "unique_key", "duplicate_key_groups", "duplicate_extra_rows", "exact_duplicate_groups", "exact_duplicate_extra_rows"]
]
show(combined_repeat_profile, "Within-source repeat evidence")

source_coverage = pd.DataFrame(
    [
        {"target entity": "orders", "JSON evidence": "orders[].header", "XML evidence": "/OperationsExport/Orders/Order/Header", "coverage": "both"},
        {"target entity": "order_items", "JSON evidence": "orders[].shoppingCart[]", "XML evidence": "/OperationsExport/Orders/Order/Shopping_Cart/Item", "coverage": "both"},
        {"target entity": "customers", "JSON evidence": "customerProfiles[]", "XML evidence": "no full customer collection", "coverage": "JSON only"},
        {"target entity": "deliveries", "JSON evidence": "orders[].delivery", "XML evidence": "/OperationsExport/Orders/Order/Delivery", "coverage": "both"},
        {"target entity": "products", "JSON evidence": "product IDs only in related records", "XML evidence": "/OperationsExport/ProductCatalogue/Product", "coverage": "XML only for full entity"},
        {"target entity": "product_reviews", "JSON evidence": "productReviews[]", "XML evidence": "/OperationsExport/ProductReviews/Review", "coverage": "both"},
        {"target entity": "warehouse reference", "JSON evidence": "warehouse name in order header", "XML evidence": "/OperationsExport/WarehouseDirectory/*", "coverage": "source-specific helper"},
    ]
)
show(source_coverage, "Source coverage and source-specific collections")

key_relationships = pd.DataFrame(
    [
        {"entity / grain": "orders / one order", "candidate primary key": "order_id", "candidate foreign keys": "customer_id -> customers.customer_id", "source evidence": "header customer and order identifiers in both sources"},
        {"entity / grain": "order_items / one line item", "candidate primary key": "order_item_id", "candidate foreign keys": "order_id -> orders.order_id | product_id -> products.product_id", "source evidence": "nested shopping-cart identifiers in both sources"},
        {"entity / grain": "customers / one customer", "candidate primary key": "customer_id", "candidate foreign keys": "none", "source evidence": "customerProfiles[].customerID"},
        {"entity / grain": "deliveries / one delivery", "candidate primary key": "delivery_id", "candidate foreign keys": "order_id -> orders.order_id", "source evidence": "nested delivery identifiers in both sources"},
        {"entity / grain": "products / one product", "candidate primary key": "product_id", "candidate foreign keys": "none in the six-table target contract", "source evidence": "/OperationsExport/ProductCatalogue/Product/Product_ID"},
        {"entity / grain": "product_reviews / one review", "candidate primary key": "review_id", "candidate foreign keys": "order_id -> orders.order_id | order_item_id -> order_items.order_item_id | product_id -> products.product_id | customer_id -> customers.customer_id", "source evidence": "review identifiers in both sources"},
        {"entity / grain": "warehouse reference / one warehouse", "candidate primary key": "Name", "candidate foreign keys": "referenced by orders.nearest_warehouse", "source evidence": "/OperationsExport/WarehouseDirectory/*/Name"},
    ]
)
show(key_relationships, "Candidate keys and relationships to verify after reconciliation")

assumptions = pd.DataFrame(
    [
        {"assumption_id": "ASM-01", "decision before transformation": "Use the published table primary key as the stable business key at each target grain; do not invent identifiers."},
        {"assumption_id": "ASM-02", "decision before transformation": "Normalise comparable types and strings before comparing duplicates or cross-source overlap."},
        {"assumption_id": "ASM-03", "decision before transformation": "Retain one canonical row when all normalised non-missing values agree; record any disagreement as validation evidence rather than applying source precedence."},
        {"assumption_id": "ASM-04", "decision before transformation": "Preserve identifier leading zeros and source case; do not lower-case structured categories."},
        {"assumption_id": "ASM-05", "decision before transformation": "Parse JSON/XML structurally before applying regex to bounded narrative fields."},
        {"assumption_id": "ASM-06", "decision before transformation": "Use literal NaN only for prescribed missing string outputs; never substitute it into required numeric, boolean, primary-key or foreign-key fields."},
        {"assumption_id": "ASM-07", "decision before transformation": "Recompute line and order arithmetic in the published sequence and use reported source amounts only as validation evidence."},
        {"assumption_id": "ASM-08", "decision before transformation": "Flatten repeated items only into their target one-to-many table; do not flatten all entities into one wide table."},
        {"assumption_id": "ASM-09", "decision before transformation": "Treat the observed counts as profiling evidence only; derive every output and validation result from the current allocated files."},
    ]
)
show(assumptions, "Pre-transformation assumption register")

assert combined_repeat_profile["missing_key"].sum() == 0, "A candidate source key is missing"
duplicate_keys_are_exact = (
    combined_repeat_profile["duplicate_key_groups"]
    == combined_repeat_profile["exact_duplicate_groups"]
).all()
assert duplicate_keys_are_exact, "A repeated business key is not an exact source duplicate"
print(f"Repeated business-key groups are exact source duplicates: {duplicate_keys_are_exact}")
print("\nTask 1 source-profile checks: PASS")



Cross-source key overlap
         entity  JSON unique keys  XML unique keys  cross-source overlap  JSON only  XML only  union before field reconciliation
         orders              2750             2750                   500       2250      2250                               5000
    order_items              8666             8622                  1577       7089      7045                              15711
     deliveries              2750             2750                   500       2250      2250                               5000
product_reviews              3850             3850                   700       3150      3150                               7000

Within-source repeat evidence
source                                 source_collection                   grain candidate_key  rows  missing_key  unique_key  duplicate_key_groups  duplicate_extra_rows  exact_duplicate_groups  exact_duplicate_extra_rows
  JSON                                customerProfiles[]    one customer pro

## 2. Source-to-target mapping

The completed field-lineage artifact is
`Group050_source_to_target_mapping.csv` in the submission root. It preserves
the template's target rows and records applicable JSON/XML structural paths,
transformation or derivation, overlap/conflict handling and stable notebook
evidence for every target field. A source path may remain blank only when that
source is not applicable, as indicated by `source_format`.

The executable audit below checks template and dictionary alignment, applicable
path completeness, actual path existence in the parsed sources, evidence IDs,
placeholder absence, unique mapping IDs and UTF-8 encoding.


In [6]:
# EVIDENCE: SEC-2-MAPPING-AUDIT
import re

mapping_template = pd.read_csv(
    MAPPING_TEMPLATE_PATH, keep_default_na=False, dtype=str, encoding="utf-8-sig"
)
mapping = pd.read_csv(
    MAPPING_OUTPUT_PATH, keep_default_na=False, dtype=str, encoding="utf-8-sig"
)
data_dictionary = pd.read_csv(
    DATA_DICTIONARY_PATH, keep_default_na=False, dtype=str, encoding="utf-8-sig"
)

expected_columns = [
    "mapping_id",
    "output_table",
    "target_field",
    "source_format",
    "json_source_path",
    "xml_source_path",
    "transformation_or_derivation",
    "overlap_or_conflict_rule",
    "notebook_evidence",
]
prefilled_columns = ["mapping_id", "output_table", "target_field"]
required_text_columns = [
    "source_format",
    "transformation_or_derivation",
    "overlap_or_conflict_rule",
    "notebook_evidence",
]
completed_columns = [
    "source_format",
    "json_source_path",
    "xml_source_path",
    "transformation_or_derivation",
    "overlap_or_conflict_rule",
    "notebook_evidence",
]
allowed_source_formats = {"JSON", "XML", "both", "derived"}
defined_evidence_ids = {
    "SEC-1.1-JSON-PROFILE",
    "SEC-1.2-XML-PROFILE",
    "SEC-1.3-SOURCE-COMPARISON",
    "SEC-2-MAPPING-AUDIT",
}

audit_rows = []

def add_mapping_check(check_id, check, observed, passed):
    audit_rows.append(
        {
            "check_id": check_id,
            "check": check,
            "observed": observed,
            "status": "PASS" if passed else "FAIL",
        }
    )

add_mapping_check(
    "MAP-AUDIT-01",
    "Required mapping filename and file presence",
    MAPPING_OUTPUT_PATH.name,
    MAPPING_OUTPUT_PATH.is_file()
    and MAPPING_OUTPUT_PATH.name == f"{GROUP_ID}_source_to_target_mapping.csv",
)
add_mapping_check(
    "MAP-AUDIT-02",
    "Exact nine-column contract",
    list(mapping.columns),
    list(mapping.columns) == expected_columns,
)
template_alignment = (
    len(mapping) == len(mapping_template)
    and mapping[prefilled_columns].equals(mapping_template[prefilled_columns])
)
add_mapping_check(
    "MAP-AUDIT-03",
    "All pre-filled template rows preserved in order",
    f"mapping={len(mapping)}, template={len(mapping_template)}",
    template_alignment,
)
dictionary_targets = data_dictionary[["output_table", "field_name"]].rename(
    columns={"field_name": "target_field"}
)
dictionary_alignment = (
    len(mapping) == len(data_dictionary)
    and mapping[["output_table", "target_field"]].equals(dictionary_targets)
)
add_mapping_check(
    "MAP-AUDIT-04",
    "Mapping targets match the public data dictionary",
    f"mapping={len(mapping)}, dictionary={len(data_dictionary)}",
    dictionary_alignment,
)
mapping_id_ok = (
    mapping["mapping_id"].is_unique
    and mapping["mapping_id"].str.fullmatch(r"MAP-[a-z_]+-\d{2}").all()
)
add_mapping_check(
    "MAP-AUDIT-05",
    "Stable and unique MAP IDs",
    f"unique={mapping['mapping_id'].nunique()}",
    mapping_id_ok,
)
source_format_ok = set(mapping["source_format"]) <= allowed_source_formats
add_mapping_check(
    "MAP-AUDIT-06",
    "Allowed source_format values only",
    sorted(mapping["source_format"].unique()),
    source_format_ok,
)
blank_required_text = [
    (row.mapping_id, column)
    for row in mapping.itertuples(index=False)
    for column in required_text_columns
    if not str(getattr(row, column)).strip()
]
add_mapping_check(
    "MAP-AUDIT-07",
    "All non-path mapping fields completed",
    f"blank cells={len(blank_required_text)}",
    not blank_required_text,
)
applicable_path_missing = []
for row in mapping.itertuples(index=False):
    json_path_present = bool(row.json_source_path.strip())
    xml_path_present = bool(row.xml_source_path.strip())
    if row.source_format in {"JSON", "both"} and not json_path_present:
        applicable_path_missing.append((row.mapping_id, "json_source_path"))
    if row.source_format in {"XML", "both"} and not xml_path_present:
        applicable_path_missing.append((row.mapping_id, "xml_source_path"))
    if row.source_format == "derived" and not (json_path_present or xml_path_present):
        applicable_path_missing.append((row.mapping_id, "derived source path"))
add_mapping_check(
    "MAP-AUDIT-08",
    "Every applicable JSON/XML path is present",
    f"missing applicable paths={len(applicable_path_missing)}",
    not applicable_path_missing,
)

json_actual_paths = set()
for prefix, records in json_collections.items():
    for record in records:
        json_actual_paths.update(f"{prefix}.{field}" for field in record)

xml_actual_paths = set()
def collect_xml_paths(element, parent_path):
    for child in element:
        child_path = f"{parent_path}/{child.tag}"
        xml_actual_paths.add(child_path)
        collect_xml_paths(child, child_path)

collect_xml_paths(xml_root, f"/{xml_root.tag}")

def split_mapping_paths(value):
    return [part.strip() for part in value.split("|") if part.strip()]

invalid_path_references = []
for row in mapping.itertuples(index=False):
    for path in split_mapping_paths(row.json_source_path):
        if path not in json_actual_paths:
            invalid_path_references.append((row.mapping_id, "JSON", path))
    for path in split_mapping_paths(row.xml_source_path):
        if path not in xml_actual_paths:
            invalid_path_references.append((row.mapping_id, "XML", path))
add_mapping_check(
    "MAP-AUDIT-09",
    "Every listed structural path exists in the parsed sources",
    f"invalid path references={len(invalid_path_references)}",
    not invalid_path_references,
)
evidence_tokens = {
    token.strip()
    for value in mapping["notebook_evidence"]
    for token in value.split("|")
    if token.strip()
}
unknown_evidence_ids = sorted(evidence_tokens - defined_evidence_ids)
all_rows_cite_mapping_audit = mapping["notebook_evidence"].str.contains(
    "SEC-2-MAPPING-AUDIT", regex=False
).all()
add_mapping_check(
    "MAP-AUDIT-10",
    "Evidence IDs are defined and every row cites this audit",
    f"unknown={unknown_evidence_ids}, all cite audit={all_rows_cite_mapping_audit}",
    not unknown_evidence_ids and all_rows_cite_mapping_audit,
)
placeholder_pattern = re.compile(r"\b(?:replace|todo|tbd|example)\b", re.IGNORECASE)
placeholder_rows = mapping[completed_columns].apply(
    lambda column: column.map(lambda value: bool(placeholder_pattern.search(value)))
).any(axis=1)
add_mapping_check(
    "MAP-AUDIT-11",
    "No template placeholders remain in completed fields",
    f"placeholder rows={int(placeholder_rows.sum())}",
    not placeholder_rows.any(),
)
utf8_bom_present = MAPPING_OUTPUT_PATH.read_bytes().startswith(b"\xef\xbb\xbf")
add_mapping_check(
    "MAP-AUDIT-12",
    "CSV is UTF-8 with BOM for portable spreadsheet display",
    f"UTF-8 BOM={utf8_bom_present}",
    utf8_bom_present,
)

mapping_audit = pd.DataFrame(audit_rows)
show(mapping_audit, "Task 1 mapping audit register")
mapping_coverage = (
    mapping.groupby(["output_table", "source_format"], sort=False)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
show(mapping_coverage, "Mapping coverage by target table and source format")
failed_mapping_checks = mapping_audit[mapping_audit["status"] != "PASS"]
assert failed_mapping_checks.empty, failed_mapping_checks.to_string(index=False)
print(f"\nFINAL_TASK1_MAPPING_AUDIT_PASS True; rows={len(mapping)}")



Task 1 mapping audit register
    check_id                                                     check                                                                                                                                                              observed status
MAP-AUDIT-01               Required mapping filename and file presence                                                                                                                                 Group050_source_to_target_mapping.csv   PASS
MAP-AUDIT-02                                Exact nine-column contract [mapping_id, output_table, target_field, source_format, json_source_path, xml_source_path, transformation_or_derivation, overlap_or_conflict_rule, notebook_evidence]   PASS
MAP-AUDIT-03           All pre-filled template rows preserved in order                                                                                                                                             mapping=111, template=111 

## Task 1 stage boundary

Task 1 ends above. The remaining sections are retained as official template
scaffolding and are intentionally not implemented or executed in this Task 1
checkpoint. They will be completed only in later stage versions.

## 3. Text and regex functions


### 3.1 Cleaning and extraction implementation


### 3.2 Public and student-designed tests


## 4. Build the six standardised relational tables

Show the transformation and row-flow evidence for each table. Keep helper
columns inside the workflow; export only fields in the public data dictionary.


### 4.1 `orders`


### 4.2 `order_items`


### 4.3 `customers`


### 4.4 `deliveries`


### 4.5 `products`


In [7]:
# EVIDENCE: SEC-4.5-PRODUCTS
# SCOPE: Build only the standardised Task 2 products table.
# This cell does not export CSV files or change later notebook sections.

from datetime import datetime
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from pathlib import Path
import html
import importlib
import re
import sys
import unicodedata

import pandas as pd


# ---------------------------------------------------------------------------
# Task 1 prerequisites
# ---------------------------------------------------------------------------

required_product_inputs = [
    "data_dictionary",
    "xml_product_records",
    "show",
]

missing_product_inputs = [
    name
    for name in required_product_inputs
    if name not in globals()
]

if missing_product_inputs:
    raise RuntimeError(
        "Run all Task 1 cells above section 4.5 first. "
        f"Missing objects: {missing_product_inputs}"
    )


# ---------------------------------------------------------------------------
# Locate and reload the shared text-function module
# ---------------------------------------------------------------------------

module_directory_candidates = [
    Path.cwd(),
    Path.cwd() / "Group050_A1",
]

if "PACKAGE_DIR" in globals():
    module_directory_candidates.insert(
        0,
        Path(PACKAGE_DIR),
    )

text_module_directory = next(
    (
        directory
        for directory in module_directory_candidates
        if (
            directory
            / "Group050_text_functions.py"
        ).is_file()
    ),
    None,
)

if text_module_directory is None:
    raise FileNotFoundError(
        "Group050_text_functions.py was not found in the notebook "
        "directory, PACKAGE_DIR, or Group050_A1."
    )

if str(text_module_directory) not in sys.path:
    sys.path.insert(0, str(text_module_directory))

import Group050_text_functions as shared_text_functions

# Reload so newly integrated teammate changes are visible without requiring
# users to close Jupyter completely.
shared_text_functions = importlib.reload(
    shared_text_functions
)

task2_text_function_usage = {}
pending_shared_text_functions = set()


# Usage: call one shared function with a private fallback.
# Reason: the uploaded module is partially implemented.
# Expected: use the shared result if implemented; otherwise use the fallback.
def _run_shared_or_private_text_function(
    function_name,
    private_function,
    value,
):
    if function_name not in pending_shared_text_functions:
        shared_function = getattr(
            shared_text_functions,
            function_name,
            None,
        )

        if callable(shared_function):
            try:
                result = shared_function(value)
            except NotImplementedError:
                pending_shared_text_functions.add(function_name)
            else:
                task2_text_function_usage[
                    function_name
                ] = "shared module"
                return result

    task2_text_function_usage[
        function_name
    ] = "private Task 2 fallback"

    return private_function(value)


# ---------------------------------------------------------------------------
# Private narrative fallback
# ---------------------------------------------------------------------------

# Usage: remove emoji from narrative text.
# Reason: emoji must not remain in product_description_clean.
# Expected: input text with common emoji ranges removed.
def _task2_remove_emoji(text):
    emoji_ranges = (
        (0x1F000, 0x1FAFF),
        (0x2600, 0x27BF),
        (0x2300, 0x23FF),
        (0x2B00, 0x2BFF),
        (0xFE00, 0xFE0F),
        (0x1F1E6, 0x1F1FF),
    )

    return "".join(
        character
        for character in text
        if character not in {"\u200d", "\u20e3"}
        and not any(
            start <= ord(character) <= end
            for start, end in emoji_ranges
        )
    )


# Usage: clean one Product_Description when the shared cleaner is pending.
# Reason: Task 2 requires product_description_clean now.
# Expected: cleaned lower-case text or literal "NaN".
def _task2_clean_narrative_text(value):
    if value is None:
        return "NaN"

    text = html.unescape(str(value))
    text = unicodedata.normalize("NFC", text)

    text = re.sub(r"<[^>]*>", " ", text)

    text = re.sub(
        r"\[(?:SYSTEM|CATALOGUE|VERIFIED_PURCHASE)\]",
        " ",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"\[SOURCE:\s*[^\]]*\]",
        " ",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"\[RATING:\s*[0-5]\s*/\s*5\]",
        " ",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"(?<![\w-])(?:#verified-buyer|@store_support)(?![\w-])",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"(?i)\b(?:https?://|www\.)\S+",
        " ",
        text,
    )

    text = _task2_remove_emoji(text)

    text = re.sub(
        (
            r"(?i)(?<![A-Z0-9])"
            r"Reference:\s*(?:HORD|CORD)\d{6}"
            r"\s*(?:[|,;/]|\s-\s)\s*"
            r"SKU:\s*SKU-[A-Z0-9]+"
            r"(?![A-Z0-9-])"
        ),
        " ",
        text,
    )

    text = re.sub(
        (
            r"(?i)(?<![A-Z0-9])"
            r"PROMO:\s*B[1-5]SAVE-\d{2}"
            r"(?![A-Z0-9-])"
        ),
        " ",
        text,
    )

    text = re.sub(r"\s+", " ", text).strip().lower()

    return text if text else "NaN"


# Usage: clean Product narrative through the shared-function adapter.
# Reason: automatically adopt the teammate implementation when completed.
# Expected: non-empty cleaned text or literal "NaN".
def _clean_task2_product_description(value):
    result = _run_shared_or_private_text_function(
        "clean_narrative_text",
        _task2_clean_narrative_text,
        value,
    )

    if not isinstance(result, str) or not result:
        raise TypeError(
            "clean_narrative_text must return non-empty text or "
            "the literal string 'NaN'."
        )

    return result


# ---------------------------------------------------------------------------
# Product field transformations
# ---------------------------------------------------------------------------

# Usage: normalise a required Product string.
# Reason: preserve identifiers/categories while removing outside whitespace.
# Expected: non-empty NFC-normalised string.
def _normalise_required_product_text(value, field_name):
    if value is None:
        raise ValueError(f"{field_name}: required text is missing")

    result = unicodedata.normalize("NFC", str(value)).strip()

    if not result:
        raise ValueError(f"{field_name}: required text is empty")

    return result


# Usage: parse an AUD-labelled Product price or cost.
# Reason: remove currency labels and thousands separators.
# Expected: float rounded to two decimal places.
def _parse_product_currency(value, field_name):
    text = _normalise_required_product_text(
        value,
        field_name,
    )
    numeric_text = re.sub(
        r"(?i)\bAUD\b",
        "",
        text,
    ).replace(",", "").strip()

    try:
        amount = Decimal(numeric_text)
    except InvalidOperation as exc:
        raise ValueError(
            f"{field_name}: invalid currency value {value!r}"
        ) from exc

    return float(
        amount.quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP,
        )
    )


# Usage: parse an integer-valued Product field.
# Reason: prevent silent rounding of invalid non-integral values.
# Expected: Python int.
def _parse_product_integer(value, field_name):
    text = _normalise_required_product_text(
        value,
        field_name,
    )

    try:
        number = Decimal(text)
    except InvalidOperation as exc:
        raise ValueError(
            f"{field_name}: invalid integer value {value!r}"
        ) from exc

    if number != number.to_integral_value():
        raise ValueError(
            f"{field_name}: expected an integer, got {value!r}"
        )

    return int(number)


# Usage: parse a numeric Product measurement.
# Reason: convert XML text while retaining published precision.
# Expected: Python float.
def _parse_product_number(value, field_name):
    text = _normalise_required_product_text(
        value,
        field_name,
    )

    try:
        return float(Decimal(text))
    except InvalidOperation as exc:
        raise ValueError(
            f"{field_name}: invalid number {value!r}"
        ) from exc


# Usage: convert a Product XML launch date.
# Reason: XML uses DD/MM/YYYY; target requires YYYY-MM-DD.
# Expected: standardised date string.
def _parse_product_date(value, field_name):
    text = _normalise_required_product_text(
        value,
        field_name,
    )

    try:
        return datetime.strptime(
            text,
            "%d/%m/%Y",
        ).strftime("%Y-%m-%d")
    except ValueError as exc:
        raise ValueError(
            f"{field_name}: invalid date {value!r}"
        ) from exc


# Usage: convert an XML Y/N Product boolean.
# Reason: target booleans must be True or False.
# Expected: Python bool.
def _parse_product_boolean(value, field_name):
    text = _normalise_required_product_text(
        value,
        field_name,
    ).upper()

    mapping = {
        "Y": True,
        "N": False,
    }

    if text not in mapping:
        raise ValueError(
            f"{field_name}: expected Y or N, got {value!r}"
        )

    return mapping[text]


# Usage: standardise one structured XML Product record.
# Reason: map source fields to the exact Product target contract.
# Expected: dictionary containing exactly 21 target fields.
def _standardise_product_record(record):
    return {
        "product_id": _normalise_required_product_text(
            record.get("Product_ID"), "Product_ID"
        ),
        "product_name": _normalise_required_product_text(
            record.get("Product_Name"), "Product_Name"
        ),
        "category": _normalise_required_product_text(
            record.get("Category"), "Category"
        ),
        "brand": _normalise_required_product_text(
            record.get("Brand"), "Brand"
        ),
        "unit_price": _parse_product_currency(
            record.get("Unit_Price"), "Unit_Price"
        ),
        "unit_cost": _parse_product_currency(
            record.get("Unit_Cost"), "Unit_Cost"
        ),
        "launch_year": _parse_product_integer(
            record.get("Launch_Year"), "Launch_Year"
        ),
        "warranty_months": _parse_product_integer(
            record.get("Warranty_Months"), "Warranty_Months"
        ),
        "weight_kg": _parse_product_number(
            record.get("Weight_Kg"), "Weight_Kg"
        ),
        "product_sku": _normalise_required_product_text(
            record.get("Product_Sku"), "Product_Sku"
        ),
        "subcategory": _normalise_required_product_text(
            record.get("Subcategory"), "Subcategory"
        ),
        "model_family": _normalise_required_product_text(
            record.get("Model_Family"), "Model_Family"
        ),
        "colour": _normalise_required_product_text(
            record.get("Colour"), "Colour"
        ),
        "supplier_id": _normalise_required_product_text(
            record.get("Supplier_ID"), "Supplier_ID"
        ),
        "supplier_country": _normalise_required_product_text(
            record.get("Supplier_Country"), "Supplier_Country"
        ),
        "launch_date": _parse_product_date(
            record.get("Launch_Date"), "Launch_Date"
        ),
        "tax_category": _normalise_required_product_text(
            record.get("Tax_Category"), "Tax_Category"
        ),
        "package_type": _normalise_required_product_text(
            record.get("Package_Type"), "Package_Type"
        ),
        "recyclable_packaging": _parse_product_boolean(
            record.get("Recyclable_Packaging"),
            "Recyclable_Packaging",
        ),
        "active_flag": _parse_product_boolean(
            record.get("Active_Flag"), "Active_Flag"
        ),
        "product_description_clean": (
            _clean_task2_product_description(
                record.get("Product_Description")
            )
        ),
    }


# Usage: reconcile normalised Product records by product_id.
# Reason: record conflicts instead of using arbitrary row precedence.
# Expected: canonical Product frame and conflict-evidence frame.
def _reconcile_product_candidates(frame):
    canonical_rows = []
    conflict_rows = []

    for product_id, group in frame.groupby(
        "product_id",
        sort=True,
        dropna=False,
    ):
        canonical = {"product_id": product_id}

        for column in frame.columns:
            if column == "product_id":
                continue

            values = [
                value
                for value in group[column].tolist()
                if not pd.isna(value)
            ]
            distinct_values = list(dict.fromkeys(values))

            if len(distinct_values) > 1:
                conflict_rows.append(
                    {
                        "product_id": product_id,
                        "field": column,
                        "normalised_values": distinct_values,
                    }
                )

            canonical[column] = (
                distinct_values[0]
                if distinct_values
                else None
            )

        canonical_rows.append(canonical)

    return (
        pd.DataFrame(
            canonical_rows,
            columns=frame.columns,
        ),
        pd.DataFrame(
            conflict_rows,
            columns=[
                "product_id",
                "field",
                "normalised_values",
            ],
        ),
    )


# ---------------------------------------------------------------------------
# Build the canonical products table
# ---------------------------------------------------------------------------

product_columns = (
    data_dictionary.loc[
        data_dictionary["output_table"].eq("products")
    ]
    .assign(
        _position=lambda frame: frame["position"].astype(int)
    )
    .sort_values("_position")["field_name"]
    .tolist()
)

product_candidates = pd.DataFrame(
    [
        _standardise_product_record(record)
        for record in xml_product_records
    ],
    columns=product_columns,
)

products, product_reconciliation_conflicts = (
    _reconcile_product_candidates(product_candidates)
)

if not product_reconciliation_conflicts.empty:
    raise ValueError(
        "Conflicting normalised Product values were found:\n"
        + product_reconciliation_conflicts
        .head(20)
        .to_string(index=False)
    )

products = (
    products.loc[:, product_columns]
    .sort_values("product_id", kind="stable")
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# Immediate Task 2 guards
# ---------------------------------------------------------------------------

if list(products.columns) != product_columns:
    raise AssertionError(
        "Product columns do not match the data dictionary."
    )

if products["product_id"].isna().any():
    raise AssertionError(
        "products.product_id contains missing values."
    )

if not products["product_id"].is_unique:
    raise AssertionError(
        "products.product_id is not unique."
    )

if products.isna().any().any():
    missing_counts = products.isna().sum()
    missing_counts = missing_counts[missing_counts.gt(0)]

    raise AssertionError(
        "Products contains prohibited missing values:\n"
        + missing_counts.to_string()
    )


# ---------------------------------------------------------------------------
# Row-flow evidence
# ---------------------------------------------------------------------------

product_row_flow = pd.DataFrame(
    [
        {
            "stage": "structured XML product records",
            "rows": len(xml_product_records),
        },
        {
            "stage": "normalised Product candidates",
            "rows": len(product_candidates),
        },
        {
            "stage": "canonical products",
            "rows": len(products),
        },
    ]
)

show(
    product_row_flow,
    "Product transformation row flow",
)
show(
    products.head(),
    "Standardised products preview",
)

print(
    "\nProduct table ready:"
    f" rows={len(products)},"
    f" columns={len(products.columns)},"
    f" reconciliation_conflicts="
    f"{len(product_reconciliation_conflicts)},"
    f" clean_narrative_text="
    f"{task2_text_function_usage.get('clean_narrative_text')}"
)


Product transformation row flow
                         stage  rows
structured XML product records  1000
 normalised Product candidates  1000
            canonical products  1000

Standardised products preview
product_id     product_name           category  brand  unit_price  unit_cost  launch_year  warranty_months  weight_kg  product_sku         subcategory model_family   colour supplier_id supplier_country launch_date tax_category package_type  recyclable_packaging  active_flag                                                                                                                        product_description_clean
   PRD0001 Candle Bloom 100             Laptop Candle     2686.14    1498.92         2014               24      1.551 SKU-CAN00001           Ultrabook          Arc Graphite      SUP001        Australia  2014-06-26 GST_STANDARD Recycled box                  True         True             ultrabook designed for portable document work and video meetings, in the arc fami

### 4.6 `product_reviews`


In [8]:
# EVIDENCE: SEC-4.6-PRODUCT-REVIEWS
# SCOPE: Build only the standardised Task 2 product_reviews table.
# Run section 4.5 first to initialise the shared-module adapter.

from datetime import datetime
from decimal import Decimal, InvalidOperation
import re
import unicodedata

import pandas as pd


# ---------------------------------------------------------------------------
# Prerequisite check
# ---------------------------------------------------------------------------

required_product_review_inputs = [
    "data_dictionary",
    "json_reviews",
    "xml_review_records",
    "show",
    "shared_text_functions",
    "_run_shared_or_private_text_function",
    "_task2_clean_narrative_text",
]

missing_product_review_inputs = [
    name
    for name in required_product_review_inputs
    if name not in globals()
]

if missing_product_review_inputs:
    raise RuntimeError(
        "Run all Task 1 cells and the updated 4.5 cell first. "
        f"Missing objects: {missing_product_review_inputs}"
    )


# ---------------------------------------------------------------------------
# Review structured-field transformations
# ---------------------------------------------------------------------------

# Usage: normalise a required Review string.
# Reason: preserve identifier/category case and leading zeros.
# Expected: non-empty NFC-normalised string.
def _normalise_required_review_text(value, field_name):
    if value is None:
        raise ValueError(f"{field_name}: required text is missing")

    result = unicodedata.normalize("NFC", str(value)).strip()

    if not result:
        raise ValueError(f"{field_name}: required text is empty")

    return result


# Usage: parse an integer Review field.
# Reason: reject malformed and non-integral numeric values.
# Expected: Python int.
def _parse_review_integer(value, field_name):
    text = _normalise_required_review_text(
        value,
        field_name,
    )

    try:
        number = Decimal(text)
    except InvalidOperation as exc:
        raise ValueError(
            f"{field_name}: invalid integer {value!r}"
        ) from exc

    if number != number.to_integral_value():
        raise ValueError(
            f"{field_name}: expected integer, got {value!r}"
        )

    return int(number)


# Usage: parse JSON/XML review timestamp.
# Reason: the two sources use different timestamp formats.
# Expected: YYYY-MM-DD HH:MM:SS string.
def _parse_review_timestamp(
    value,
    field_name,
    source_format,
):
    text = _normalise_required_review_text(
        value,
        field_name,
    )

    formats = {
        "JSON": "%Y-%m-%d %H:%M:%S",
        "XML": "%d/%m/%Y %H:%M:%S",
    }

    if source_format not in formats:
        raise ValueError(
            f"Unsupported source format: {source_format}"
        )

    try:
        parsed = datetime.strptime(
            text,
            formats[source_format],
        )
    except ValueError as exc:
        raise ValueError(
            f"{field_name}: invalid timestamp {value!r}"
        ) from exc

    return parsed.strftime("%Y-%m-%d %H:%M:%S")


# Usage: normalise JSON bool or XML Y/N.
# Reason: target requires True/False.
# Expected: Python bool.
def _parse_review_boolean(value, field_name):
    if isinstance(value, bool):
        return value

    text = _normalise_required_review_text(
        value,
        field_name,
    ).upper()

    mapping = {
        "Y": True,
        "N": False,
        "TRUE": True,
        "FALSE": False,
    }

    if text not in mapping:
        raise ValueError(
            f"{field_name}: invalid boolean {value!r}"
        )

    return mapping[text]


# ---------------------------------------------------------------------------
# Private fallbacks for functions still pending in the shared module
# ---------------------------------------------------------------------------

# Usage: extract a raw order reference.
# Reason: the current shared function is pending.
# Expected: upper-case HORD/CORD plus six digits or literal "NaN".
def _task2_extract_order_reference(value):
    if value is None:
        return "NaN"

    match = re.search(
        (
            r"(?i)(?<![A-Z0-9])"
            r"(?:HORD|CORD)\d{6}"
            r"(?![A-Z0-9])"
        ),
        str(value),
    )

    return match.group(0).upper() if match else "NaN"


# Usage: extract a raw Product SKU.
# Reason: provide a fallback if the shared function becomes unavailable.
# Expected: upper-case bounded SKU or literal "NaN".
def _task2_extract_product_sku(value):
    if value is None:
        return "NaN"

    match = re.search(
        (
            r"(?i)(?<![A-Z0-9-])"
            r"SKU-[A-Z0-9]+"
            r"(?![A-Z0-9-])"
        ),
        str(value),
    )

    return match.group(0).upper() if match else "NaN"


# Usage: identify a Latin-script letter.
# Reason: accented Latin characters are not non-Latin merely because non-ASCII.
# Expected: Python bool.
def _task2_is_latin_letter(character):
    return (
        unicodedata.category(character).startswith("L")
        and "LATIN" in unicodedata.name(character, "")
    )


# Usage: derive Latin analysis from review_body_clean.
# Reason: the current shared implementation is pending.
# Expected: Latin-analysis text or literal "NaN".
def _task2_build_latin_analysis(value):
    if value is None or value == "NaN":
        return "NaN"

    text = unicodedata.normalize("NFC", str(value))
    result_characters = []
    previous_base_is_latin = False

    for character in text:
        category = unicodedata.category(character)

        if category.startswith("L"):
            if _task2_is_latin_letter(character):
                result_characters.append(character)
                previous_base_is_latin = True
            else:
                result_characters.append(" ")
                previous_base_is_latin = False

        elif category.startswith("M"):
            if previous_base_is_latin:
                result_characters.append(character)

        else:
            result_characters.append(character)
            previous_base_is_latin = False

    result = re.sub(
        r"\s+",
        " ",
        "".join(result_characters),
    ).strip()

    contains_latin = any(
        _task2_is_latin_letter(character)
        for character in result
    )

    return result if contains_latin else "NaN"


# Usage: identify non-Latin letters in review_body_clean.
# Reason: the current shared implementation is pending.
# Expected: True when a non-Latin letter exists; otherwise False.
def _task2_contains_non_latin_script(value):
    if value is None or value == "NaN":
        return False

    text = unicodedata.normalize("NFC", str(value))

    return any(
        unicodedata.category(character).startswith("L")
        and not _task2_is_latin_letter(character)
        for character in text
    )


# ---------------------------------------------------------------------------
# Text-function wrappers
# ---------------------------------------------------------------------------

def _clean_product_review_text(value):
    result = _run_shared_or_private_text_function(
        "clean_narrative_text",
        _task2_clean_narrative_text,
        value,
    )

    if not isinstance(result, str) or not result:
        raise TypeError(
            "clean_narrative_text returned an invalid value."
        )

    return result


def _extract_product_review_order_reference(value):
    result = _run_shared_or_private_text_function(
        "extract_order_reference",
        _task2_extract_order_reference,
        value,
    )

    if not isinstance(result, str) or not result:
        raise TypeError(
            "extract_order_reference returned an invalid value."
        )

    return result


def _extract_product_review_sku(value):
    result = _run_shared_or_private_text_function(
        "extract_product_sku",
        _task2_extract_product_sku,
        value,
    )

    if not isinstance(result, str) or not result:
        raise TypeError(
            "extract_product_sku returned an invalid value."
        )

    return result


def _build_product_review_latin_analysis(value):
    result = _run_shared_or_private_text_function(
        "build_latin_analysis",
        _task2_build_latin_analysis,
        value,
    )

    if not isinstance(result, str) or not result:
        raise TypeError(
            "build_latin_analysis returned an invalid value."
        )

    return result


def _product_review_contains_non_latin(value):
    result = _run_shared_or_private_text_function(
        "contains_non_latin_script",
        _task2_contains_non_latin_script,
        value,
    )

    if not isinstance(result, bool):
        raise TypeError(
            "contains_non_latin_script must return bool."
        )

    return result


# ---------------------------------------------------------------------------
# Source transformations
# ---------------------------------------------------------------------------

# Usage: derive all text-dependent target fields from raw review text.
# Reason: ensure references are extracted before cleaning.
# Expected: dictionary containing all derived review text fields.
def _derive_product_review_text_fields(raw_review_text):
    extracted_order_reference = (
        _extract_product_review_order_reference(
            raw_review_text
        )
    )
    extracted_product_sku = (
        _extract_product_review_sku(
            raw_review_text
        )
    )

    review_body_clean = _clean_product_review_text(
        raw_review_text
    )
    review_body_latin_analysis = (
        _build_product_review_latin_analysis(
            review_body_clean
        )
    )
    contains_non_latin_script = (
        _product_review_contains_non_latin(
            review_body_clean
        )
    )

    return {
        "review_body_clean": review_body_clean,
        "review_body_latin_analysis": (
            review_body_latin_analysis
        ),
        "review_length_chars": (
            0
            if review_body_clean == "NaN"
            else len(review_body_clean)
        ),
        "review_word_count": (
            0
            if review_body_clean == "NaN"
            else len(review_body_clean.split())
        ),
        "contains_non_latin_script": (
            contains_non_latin_script
        ),
        "extracted_order_reference": (
            extracted_order_reference
        ),
        "extracted_product_sku": extracted_product_sku,
    }


# Usage: map one JSON Review to the target.
# Reason: handle JSON-specific field names and representations.
# Expected: dictionary containing exactly 21 target fields.
def _standardise_json_product_review(record):
    derived = _derive_product_review_text_fields(
        record.get("reviewText")
    )

    return {
        "review_id": _normalise_required_review_text(
            record.get("reviewID"), "reviewID"
        ),
        "order_id": _normalise_required_review_text(
            record.get("orderID"), "orderID"
        ),
        "order_item_id": _normalise_required_review_text(
            record.get("orderItemID"), "orderItemID"
        ),
        "product_id": _normalise_required_review_text(
            record.get("productID"), "productID"
        ),
        "customer_id": _normalise_required_review_text(
            record.get("customerID"), "customerID"
        ),
        "review_timestamp": _parse_review_timestamp(
            record.get("reviewTimestamp"),
            "reviewTimestamp",
            "JSON",
        ),
        "language_code": _normalise_required_review_text(
            record.get("languageCode"), "languageCode"
        ),
        "rating": _parse_review_integer(
            record.get("rating"), "rating"
        ),
        "review_title": _normalise_required_review_text(
            record.get("reviewTitle"), "reviewTitle"
        ),
        "review_body_clean": derived[
            "review_body_clean"
        ],
        "review_body_latin_analysis": derived[
            "review_body_latin_analysis"
        ],
        "verified_purchase": _parse_review_boolean(
            record.get("verifiedPurchase"),
            "verifiedPurchase",
        ),
        "helpful_votes": _parse_review_integer(
            record.get("helpfulVotes"), "helpfulVotes"
        ),
        "review_length_chars": derived[
            "review_length_chars"
        ],
        "review_word_count": derived[
            "review_word_count"
        ],
        "contains_non_latin_script": derived[
            "contains_non_latin_script"
        ],
        "extracted_order_reference": derived[
            "extracted_order_reference"
        ],
        "extracted_product_sku": derived[
            "extracted_product_sku"
        ],
        "delivery_experience": _normalise_required_review_text(
            record.get("deliveryExperience"),
            "deliveryExperience",
        ),
        "value_experience": _normalise_required_review_text(
            record.get("valueExperience"),
            "valueExperience",
        ),
        "writing_style": _normalise_required_review_text(
            record.get("writingStyle"),
            "writingStyle",
        ),
    }


# Usage: map one XML Review to the target.
# Reason: handle XML-specific names, date format and boolean representation.
# Expected: dictionary containing exactly 21 target fields.
def _standardise_xml_product_review(record):
    derived = _derive_product_review_text_fields(
        record.get("Review_Text")
    )

    return {
        "review_id": _normalise_required_review_text(
            record.get("Review_ID"), "Review_ID"
        ),
        "order_id": _normalise_required_review_text(
            record.get("Order_ID"), "Order_ID"
        ),
        "order_item_id": _normalise_required_review_text(
            record.get("Order_Item_ID"), "Order_Item_ID"
        ),
        "product_id": _normalise_required_review_text(
            record.get("Product_ID"), "Product_ID"
        ),
        "customer_id": _normalise_required_review_text(
            record.get("Customer_ID"), "Customer_ID"
        ),
        "review_timestamp": _parse_review_timestamp(
            record.get("Review_Timestamp"),
            "Review_Timestamp",
            "XML",
        ),
        "language_code": _normalise_required_review_text(
            record.get("Language_Code"), "Language_Code"
        ),
        "rating": _parse_review_integer(
            record.get("Rating"), "Rating"
        ),
        "review_title": _normalise_required_review_text(
            record.get("Review_Title"), "Review_Title"
        ),
        "review_body_clean": derived[
            "review_body_clean"
        ],
        "review_body_latin_analysis": derived[
            "review_body_latin_analysis"
        ],
        "verified_purchase": _parse_review_boolean(
            record.get("Verified_Purchase"),
            "Verified_Purchase",
        ),
        "helpful_votes": _parse_review_integer(
            record.get("Helpful_Votes"),
            "Helpful_Votes",
        ),
        "review_length_chars": derived[
            "review_length_chars"
        ],
        "review_word_count": derived[
            "review_word_count"
        ],
        "contains_non_latin_script": derived[
            "contains_non_latin_script"
        ],
        "extracted_order_reference": derived[
            "extracted_order_reference"
        ],
        "extracted_product_sku": derived[
            "extracted_product_sku"
        ],
        "delivery_experience": _normalise_required_review_text(
            record.get("Delivery_Experience"),
            "Delivery_Experience",
        ),
        "value_experience": _normalise_required_review_text(
            record.get("Value_Experience"),
            "Value_Experience",
        ),
        "writing_style": _normalise_required_review_text(
            record.get("Writing_Style"),
            "Writing_Style",
        ),
    }


# ---------------------------------------------------------------------------
# Build source candidates
# ---------------------------------------------------------------------------

product_review_columns = (
    data_dictionary.loc[
        data_dictionary["output_table"].eq("product_reviews")
    ]
    .assign(
        _position=lambda frame: frame["position"].astype(int)
    )
    .sort_values("_position")["field_name"]
    .tolist()
)

json_product_review_candidates = pd.DataFrame(
    [
        _standardise_json_product_review(record)
        for record in json_reviews
    ],
    columns=product_review_columns,
)
json_product_review_candidates["_source"] = "JSON"

xml_product_review_candidates = pd.DataFrame(
    [
        _standardise_xml_product_review(record)
        for record in xml_review_records
    ],
    columns=product_review_columns,
)
xml_product_review_candidates["_source"] = "XML"

product_review_candidates = pd.concat(
    [
        json_product_review_candidates,
        xml_product_review_candidates,
    ],
    ignore_index=True,
)


# ---------------------------------------------------------------------------
# Reconciliation
# ---------------------------------------------------------------------------

# Usage: reconcile normalised reviews by review_id.
# Reason: avoid arbitrary JSON/XML precedence.
# Expected: canonical reviews and conflict-evidence frame.
def _reconcile_product_review_candidates(
    frame,
    output_columns,
):
    canonical_rows = []
    conflict_rows = []

    for review_id, group in frame.groupby(
        "review_id",
        sort=True,
        dropna=False,
    ):
        canonical = {"review_id": review_id}
        sources = sorted(group["_source"].unique())

        for column in output_columns:
            if column == "review_id":
                continue

            values = [
                value
                for value in group[column].tolist()
                if not pd.isna(value)
            ]
            distinct_values = list(dict.fromkeys(values))

            if len(distinct_values) > 1:
                conflict_rows.append(
                    {
                        "review_id": review_id,
                        "field": column,
                        "normalised_values": distinct_values,
                        "sources": sources,
                    }
                )

            canonical[column] = (
                distinct_values[0]
                if distinct_values
                else None
            )

        canonical_rows.append(canonical)

    return (
        pd.DataFrame(
            canonical_rows,
            columns=output_columns,
        ),
        pd.DataFrame(
            conflict_rows,
            columns=[
                "review_id",
                "field",
                "normalised_values",
                "sources",
            ],
        ),
    )


product_reviews, product_review_reconciliation_conflicts = (
    _reconcile_product_review_candidates(
        product_review_candidates,
        product_review_columns,
    )
)

if not product_review_reconciliation_conflicts.empty:
    raise ValueError(
        "Conflicting Product Review values were found. "
        f"Total conflicts="
        f"{len(product_review_reconciliation_conflicts)}\n"
        + product_review_reconciliation_conflicts
        .head(20)
        .to_string(index=False)
    )

product_reviews = (
    product_reviews.loc[:, product_review_columns]
    .sort_values("review_id", kind="stable")
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# Immediate Task 2 guards
# ---------------------------------------------------------------------------

if list(product_reviews.columns) != product_review_columns:
    raise AssertionError(
        "Product Review columns do not match the dictionary."
    )

if product_reviews["review_id"].isna().any():
    raise AssertionError(
        "product_reviews.review_id contains missing values."
    )

if not product_reviews["review_id"].is_unique:
    raise AssertionError(
        "product_reviews.review_id is not unique."
    )

if product_reviews.isna().any().any():
    missing_counts = product_reviews.isna().sum()
    missing_counts = missing_counts[missing_counts.gt(0)]

    raise AssertionError(
        "Product Reviews contains prohibited missing values:\n"
        + missing_counts.to_string()
    )

if not pd.api.types.is_bool_dtype(
    product_reviews["verified_purchase"]
):
    raise AssertionError(
        "verified_purchase is not boolean."
    )

if not pd.api.types.is_bool_dtype(
    product_reviews["contains_non_latin_script"]
):
    raise AssertionError(
        "contains_non_latin_script is not boolean."
    )

for integer_column in [
    "rating",
    "helpful_votes",
    "review_length_chars",
    "review_word_count",
]:
    if not pd.api.types.is_integer_dtype(
        product_reviews[integer_column]
    ):
        raise AssertionError(
            f"{integer_column} is not integer-valued."
        )


# ---------------------------------------------------------------------------
# Row-flow and reconciliation evidence
# ---------------------------------------------------------------------------

json_review_ids = set(
    json_product_review_candidates["review_id"]
)
xml_review_ids = set(
    xml_product_review_candidates["review_id"]
)

product_review_row_flow = pd.DataFrame(
    [
        {
            "stage": "structured JSON review records",
            "rows": len(json_reviews),
        },
        {
            "stage": "normalised JSON candidates",
            "rows": len(json_product_review_candidates),
        },
        {
            "stage": "structured XML review records",
            "rows": len(xml_review_records),
        },
        {
            "stage": "normalised XML candidates",
            "rows": len(xml_product_review_candidates),
        },
        {
            "stage": "combined candidates",
            "rows": len(product_review_candidates),
        },
        {
            "stage": "canonical product_reviews",
            "rows": len(product_reviews),
        },
    ]
)

product_review_overlap_summary = pd.DataFrame(
    [
        {
            "JSON unique review IDs": len(json_review_ids),
            "XML unique review IDs": len(xml_review_ids),
            "cross-source overlap": len(
                json_review_ids & xml_review_ids
            ),
            "JSON duplicate ID groups": int(
                json_product_review_candidates[
                    "review_id"
                ]
                .value_counts()
                .gt(1)
                .sum()
            ),
            "XML duplicate ID groups": int(
                xml_product_review_candidates[
                    "review_id"
                ]
                .value_counts()
                .gt(1)
                .sum()
            ),
            "reconciliation conflicts": len(
                product_review_reconciliation_conflicts
            ),
        }
    ]
)

preview_columns = [
    "review_id",
    "order_id",
    "order_item_id",
    "product_id",
    "customer_id",
    "review_timestamp",
    "language_code",
    "rating",
    "verified_purchase",
    "helpful_votes",
    "review_length_chars",
    "review_word_count",
    "contains_non_latin_script",
    "extracted_order_reference",
    "extracted_product_sku",
]

show(
    product_review_row_flow,
    "Product Review transformation row flow",
)
show(
    product_review_overlap_summary,
    "Product Review duplicate and overlap summary",
)
show(
    product_reviews.loc[:, preview_columns].head(),
    "Standardised product_reviews preview",
)

print("\nTask 2 text-function integration:")
for function_name in [
    "clean_narrative_text",
    "extract_order_reference",
    "extract_product_sku",
    "build_latin_analysis",
    "contains_non_latin_script",
]:
    print(
        f"- {function_name}: "
        f"{task2_text_function_usage.get(function_name)}"
    )

print(
    "\nProduct Review table ready:"
    f" rows={len(product_reviews)},"
    f" columns={len(product_reviews.columns)},"
    f" reconciliation_conflicts="
    f"{len(product_review_reconciliation_conflicts)}"
)


Product Review transformation row flow
                         stage  rows
structured JSON review records  3946
    normalised JSON candidates  3946
 structured XML review records  3946
     normalised XML candidates  3946
           combined candidates  7892
     canonical product_reviews  7000

Product Review duplicate and overlap summary
 JSON unique review IDs  XML unique review IDs  cross-source overlap  JSON duplicate ID groups  XML duplicate ID groups  reconciliation conflicts
                   3850                   3850                   700                        96                       96                         0

Standardised product_reviews preview
 review_id   order_id order_item_id product_id customer_id    review_timestamp language_code  rating  verified_purchase  helpful_votes  review_length_chars  review_word_count  contains_non_latin_script extracted_order_reference extracted_product_sku
HREV000001 HORD000001   HITM0000001    PRD0837    CUS00191 2018-12-16 11:01

## 5. Reconcile overlap and verify relationships

Demonstrate how records are compared by stable business key, how canonical
rows are retained and how silent source-precedence choices are avoided.


## 6. Validation register

Keep each check executable and give it a stable `VAL-...` ID. Immediately after
each code check, record the observed result, `PASS`/`FAIL`, evidence and
resolution/interpretation. A genuine, explained failure is preferable to a
fabricated pass.

Required areas include schema/types, primary and foreign keys, row flow and
source coverage, overlap, arithmetic, temporal logic, text/reference behaviour
and multilingual handling.


### 6.1 Schema and type checks (`VAL-SCHEMA-...`)


**Observed result/status/interpretation:** Replace.


### 6.2 Primary- and foreign-key checks (`VAL-PK-...`, `VAL-FK-...`)


**Observed result/status/interpretation:** Replace.


### 6.3 Source coverage and reconciliation checks (`VAL-FLOW-...`)


**Observed result/status/interpretation:** Replace.


### 6.4 Arithmetic checks (`VAL-ARITH-...`)


**Observed result/status/interpretation:** Replace.


### 6.5 Temporal checks (`VAL-TIME-...`)


**Observed result/status/interpretation:** Replace.


### 6.6 Text and multilingual checks (`VAL-TEXT-...`)


**Observed result/status/interpretation:** Replace.


### 6.7 Literal `NaN` reminder

For prescribed missing string outputs, the expected value is the three text
characters `NaN`, not an empty field, Python `None` or a floating-point NaN.
Use `pandas.read_csv(path, keep_default_na=False)` when validating that sentinel.


## 7. Export the six CSV files

Export exactly the required filenames, columns and order. Display a compact
final schema/row-count summary without hard-coding certified counts.


## 8. Final reproducibility record

Record the final run date, dependency versions and the result of Restart and Run
All. Confirm that the six outputs and validation evidence were recreated from
the allocated raw files.
